<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 151 (delta 79), reused 6 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 58.50 KiB | 820.00 KiB/s, done.
Resolving deltas: 100% (79/79), done.


In [24]:
%cd /content/ML-Tech

/content/ML-Tech


In [25]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 16 chunks
{'document': 'Personal attendance required.txt', 'section': 'Personal attendance required', 'text': 'Category:'}


In [26]:
!pip install -q sentence-transformers faiss-cpu

In [27]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [28]:
texts = [chunk["text"] for chunk in chunks]
passages = ["passage: " + text for text in texts]

In [29]:
embeddings = model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(16, 768)


In [30]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [31]:
question="What documents are required for a biometric passport?"
question_embedding = model.encode(["query:" + question], normalize_embeddings=True)

In [32]:
k = 3

scores, indices = index.search(question_embedding, k)
for rank, idx in enumerate(indices[0]):
    print("=" * 80)
    print("Rank:", rank + 1)
    print("Score:", scores[0][rank])
    print("Document:", chunks[idx]["document"])
    print("Section:", chunks[idx]["section"])
    print("Text:")
    print(chunks[idx]["text"])

Rank: 1
Score: 0.8574393
Document: Biometric Passport.txt
Section: Full Document
Text:
URL: https://www.general-security.gov.lb/en/posts/11 
Title: Biometric Passport
Category:Biometric Passport
Keywords: documents, fees, minors, renewal, biometric
Content:
Requested documents:

The adequate application for passports format A4 (10 years) issued by the competent mayor according to the place of residence.
Lebanese ID card OR/AND an extract of civil status (whether the Lebanese citizen is applying for the 1st time for a biometric passport or not). Follow this link for more information: https://www.general-security.gov.lb/ar/posts/408
A new colored photo ID photo on a white background, 4.5 x 3.5, on which the name of the individual appears, as well as the number and place of registered residence, signed and certified by the mayor.
The old passport if the latter is available, as well as a copy of the pages that are not empty.
The fees related to this application.
When it comes to members of